# Lab 2 — Classification Models

**DA380A Machine Learning · Group 28**

**Dataset:** `CarData` (`ALL_CARS_DATA.csv`) — 266,847 car listings scraped from cars.com: brand, origin, car type, age, mileage, stock type (new/used/certified), customer rating, number of reviews, dealer location and price.

**Target:** `PRICE RANGE` — the listing's price class: **Low, Medium, High or Luxury** (4 classes). As section 1.3 shows, it is effectively *how expensive the car is compared with other cars of the same type*. Predicting it tells a buyer or dealer which price tier a car belongs in **without knowing its price**.

Why this dataset, and every cleaning decision, is documented in [`docs/step1-dataset-choice.md`](../docs/step1-dataset-choice.md).

| Section | Lab task |
|---|---|
| 1 | Explore and understand the dataset |
| 2 | Prepare the data |
| 3 | Create training and test data |
| 4 | Build at least two classification models |
| 5 | Evaluate the models |
| 6 | Improve and tune the models |
| 7 | Interpret and compare the results |

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.overview import compare_datasets, load_dataset
from src.data import (CATEGORICAL, CLASS_ORDER, CSV_PATH, NUMERIC, RANDOM_STATE, SKEWED_NUMERIC,
                      TARGET, load_cars, load_raw_with_price)

SAMPLE_SIZE = 40_000   # lower to 20_000 for a faster run while experimenting
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Explore and understand the dataset

### 1.1 Choosing the dataset
We compared the four provided datasets on: a clear class target, enough rows, useful features, data quality, class balance and leakage risk (this cell reads all four files and takes about a minute).

In [ ]:
overview = compare_datasets()
overview

`CarData` has a ready-made, **balanced 4-class target** and a good mix of numeric and categorical features. The alternatives were harder: `NetworkEvent` is 96% / 4% imbalanced with 60% missing values in one column and inconsistent labels (`PolicyViolation` vs `Policy_Violation`); `musicdata` has no natural class to predict; `perfumes` would need its main feature (a list of scent accords stored as text) parsed first. Full comparison: `docs/step1-dataset-choice.md`.

### 1.2 First look at the raw data

In [ ]:
raw = load_dataset(CSV_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
print("Missing values per column:")
print(raw.isna().sum()[lambda s: s > 0])
print("\nFully duplicated rows:", raw.duplicated().sum())
content_columns = [c for c in raw.columns if c not in ("CAR ID", "IMAGE")]
print("Rows identical apart from CAR ID / IMAGE:", raw.duplicated(subset=content_columns).sum())
print("\nUnique values:", raw.nunique().to_dict())

### 1.3 Data-quality and leakage checks

**Hidden missing values.** `RATING` and `REVIEW` have no `NaN`s, but their minimum is **−1** — a code for "no rating / no reviews yet", not a real value. (`AGE OF CAR = −1` *is* real: it's computed as 2024 − model year, so 2025 models get −1.)

In [ ]:
for col in ["RATING", "REVIEW", "AGE OF CAR"]:
    print(f"{col:12s} min = {raw[col].min():>6}   rows equal to -1: {(raw[col] == -1).sum():,}")
print("\nAGE OF CAR == 2024 - MODEL for every row:", ((2024 - raw["MODEL"]) == raw["AGE OF CAR"]).all())

**Inconsistent categories.** The same country is spelled two ways, and `STOCK TYPE` has 35 values — `New`, `Used` and 33 brand-specific *certified pre-owned* labels:

In [ ]:
print(raw["MAKE ORIGIN"].value_counts().to_dict())
print()
print(raw["STOCK TYPE"].value_counts().head(8).to_dict(), "...")

**Leakage.** `PRICE RANGE` is a label made from the price, so `PRICE($)` must not be a feature. But the price ranges *overlap* between classes — a "Low" car can cost more than a "Luxury" one — so it isn't one fixed cut-off for all cars:

In [ ]:
raw.groupby(TARGET)["PRICE($)"].describe()[["min", "50%", "max"]].reindex(CLASS_ORDER)

In [ ]:
# Hypothesis: the class is the price quartile *within each car type*
labelled = raw.dropna(subset=[TARGET])
price_rank_in_type = labelled.groupby("CAR TYPE")["PRICE($)"].rank(pct=True)
quartile_rule = pd.cut(price_rank_in_type, [0, 0.25, 0.5, 0.75, 1.0], labels=CLASS_ORDER)
print(f"Price quartile within CAR TYPE reproduces PRICE RANGE for "
      f"{(quartile_rule.astype(str) == labelled[TARGET]).mean():.1%} of rows")

big_types = labelled["CAR TYPE"].value_counts().head(6).index
pd.crosstab(labelled.loc[labelled["CAR TYPE"].isin(big_types), "CAR TYPE"],
            labelled.loc[labelled["CAR TYPE"].isin(big_types), TARGET],
            normalize="index").reindex(columns=CLASS_ORDER).round(2)

A simple rule — *rank the price among cars of the same type and cut it into quarters* — reproduces the label for most rows, and inside each big car type the four classes are about 25% each. So **`PRICE RANGE` is essentially the car's price compared with other cars of its type.** With `PRICE($)` as a feature the task would be trivial (section 7.4), so it is removed. That also means `CAR TYPE` on its own says almost nothing about the class.

### 1.4 Cleaning
All cleaning lives in `src/data.py`; each step prints what it changed:

In [ ]:
df, notes = load_cars(sample_size=SAMPLE_SIZE)
print("\n".join(notes))
print("\nCleaned dataset:", df.shape)
df.head()

In [ ]:
df.describe().T

### 1.5 The target: class balance

In [ ]:
class_share = df[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER)
ax = class_share.plot(kind="bar", figsize=(6, 3.5), rot=0)
ax.set(title="PRICE RANGE — share of rows per class", ylabel="Share")
for i, v in enumerate(class_share):
    ax.text(i, v + 0.005, f"{v:.1%}", ha="center")
plt.tight_layout(); plt.show()

### 1.6 Numeric features per class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
plot_specs = [("AGE OF CAR", False), ("MILEAGE", True), ("RATING", False), ("REVIEW", True)]
for ax, (col, use_log) in zip(axes, plot_specs):
    values = np.log1p(df[col]) if use_log else df[col]
    sns.boxplot(x=df[TARGET], y=values, order=CLASS_ORDER, ax=ax, showfliers=False)
    ax.set(title=f"{'log(1 + ' + col + ')' if use_log else col} by class", xlabel="", ylabel="")
plt.tight_layout(); plt.show()

df.groupby(TARGET)[SKEWED_NUMERIC + NUMERIC].median().reindex(CLASS_ORDER)

### 1.7 Categorical features per class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

stock = pd.crosstab(df["STOCK TYPE"], df[TARGET], normalize="index").reindex(columns=CLASS_ORDER)
stock.plot(kind="bar", stacked=True, ax=axes[0], rot=0)
axes[0].set(title="Class mix per stock type", ylabel="Share")
axes[0].legend(title="PRICE RANGE", bbox_to_anchor=(1, 1))

common_brands = df["BRAND"].value_counts().loc[lambda s: s >= 300].index
brand = pd.crosstab(df["BRAND"], df[TARGET], normalize="index").reindex(columns=CLASS_ORDER).loc[common_brands]
brand.sort_values("Luxury").plot(kind="barh", stacked=True, ax=axes[1], legend=False)
axes[1].set(title="Class mix per brand (brands with 300+ listings)", xlabel="Share")
plt.tight_layout(); plt.show()

stock.round(2)

### 1.8 Main observations

- After cleaning: **240,927 unique listings**; we work with a random sample of **40,000** (9 features + target) to keep tuning fast.
- **25,919 listings were duplicates** that differed only in `CAR ID` / `IMAGE` (the same car scraped twice) — removed so a car can't appear in both the training and the test set.
- **`RATING` was missing for ≈ 20% of rows**, hidden as the code −1; it is now a real missing value.
- The **classes are balanced** (24–26% each), so accuracy is a meaningful metric and no resampling is needed.
- **`PRICE RANGE` is the price compared within the car type** → `PRICE($)` is leakage and is removed, and `CAR TYPE` alone carries little information.
- **Stock type is the strongest single signal**: used cars are 45% *Low* and only 10% *Luxury*; new cars are 44% *Luxury* and only 5% *Low*.
- **Age and mileage separate the cheap classes**: *Low* cars have a median age of 6 years and ≈ 74,000 miles, *Medium* 2 years and ≈ 22,500 miles. But **_High_ and _Luxury_ cars are both mostly new** (median age 0, mileage 0) — we expect those two classes to be the hardest to tell apart.
- **Brand matters**: Volvo and Porsche listings are mostly *Luxury* (≈ 55–60%), Hyundai and Kia almost never (≈ 5%).

## 2. Prepare the data

**Features (X):** `MAKE ORIGIN`, `BRAND`, `CAR TYPE`, `STOCK TYPE`, `DEALER LOCATION (STATE)` (categorical) and `MILEAGE`, `REVIEW`, `AGE OF CAR`, `RATING` (numeric). **Target (y):** `PRICE RANGE`.

Already done in `load_cars()` (section 1.4), because these steps don't learn anything from the data: removed duplicates and the one unlabelled row, recoded −1 as missing, unified country spellings, grouped stock types, and dropped `PRICE($)` (leakage), the ID columns, `MODEL` (= 2024 − age), `PARENT COMPANY` (≈ brand) and the very high-cardinality text columns (model name, dealer name, city).

Steps that **learn from the data** go into a `Pipeline`, so they are fitted on the training data only — inside every cross-validation fold too.

| Column type | Step | Why |
|---|---|---|
| Numeric | `SimpleImputer(median, add_indicator=True)` | Fills the missing ratings with the median, and adds a 0/1 column saying *the rating was missing* — "no rating yet" may itself be informative |
| `MILEAGE`, `REVIEW` | `log1p` | Long right tails (mileage up to ≈ 480,000); the log stops a few extreme values from dominating |
| Numeric | `StandardScaler` | Logistic regression is sensitive to feature scale; the random forest doesn't need it but isn't hurt |
| Categorical | `SimpleImputer(most_frequent)` | Safety net for missing categories |
| Categorical | `OneHotEncoder(handle_unknown="ignore")` | Brand, type, origin and state have no natural order; unseen categories in new data won't crash the model |

In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]
print("X:", X.shape, "| y:", y.shape)
X.dtypes

In [ ]:
skewed_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("skewed", skewed_pipeline, SKEWED_NUMERIC),
    ("num", numeric_pipeline, NUMERIC),
    ("cat", categorical_pipeline, CATEGORICAL),
])
preprocess

## 3. Create training and test data

80% training, 20% test, **stratified** on the class so every class keeps the same share in both sets. The split happens **before** anything is fitted, and the test set isn't used again until section 5. Model comparison and tuning use **5-fold (stratified) cross-validation on the training set**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
pd.DataFrame({"train": y_train.value_counts(normalize=True),
              "test": y_test.value_counts(normalize=True)}).reindex(CLASS_ORDER).round(3)

In [ ]:
# Check the preprocessing on the training set only
X_train_prepared = preprocess.fit_transform(X_train)
print("Features after preprocessing:", X_train_prepared.shape[1])
print("Missing values after preprocessing:", np.isnan(X_train_prepared).sum())

## 4. Build the classification models

| Model | Why |
|---|---|
| **Dummy (most frequent)** | Always predicts the largest class. Any useful model must beat it (≈ 26% accuracy here) |
| **Logistic Regression** | Linear and interpretable: one weight per feature and class, outputs class probabilities. With 4 classes it uses the *softmax* (multinomial) version |
| **Random Forest** | Many decision trees, each voting for a class. Captures non-linear effects and interactions (e.g. a brand that is luxury only when the car is new) |

**Metrics:** accuracy plus **macro-averaged** precision, recall and F1 — the average of the per-class scores, so every class counts equally. Because the classes are balanced, macro and weighted averages are almost the same; macro fits the question *"which classes are hard?"* best. All models are compared with 5-fold cross-validation on the training set.

In [ ]:
def build_pipeline(model):
    """Fresh copy of the preprocessing + a fresh copy of the model, so pipelines never share fitted state."""
    return Pipeline([("prep", clone(preprocess)), ("model", clone(model))])


models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
}
scoring = {"Accuracy": "accuracy", "Precision": "precision_macro",
           "Recall": "recall_macro", "F1": "f1_macro"}

cv_rows = []
for name, model in models.items():
    cv = cross_validate(build_pipeline(model), X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    cv_rows.append({"Model": name, **{f"CV {m}": cv[f"test_{m}"].mean() for m in scoring},
                    "CV F1 std": cv["test_F1"].std()})

cv_results = pd.DataFrame(cv_rows).set_index("Model").round(3)
cv_results

## 5. Evaluate the models on the test set

Each model is fitted on the full training set and evaluated **once** on the same test set.

- **Accuracy** — share of all cars put in the right class.
- **Precision** (per class) — of the cars predicted as *Luxury*, how many really are? **Recall** — of the real *Luxury* cars, how many did we find? **F1** combines both.
- **Confusion matrix** — rows are the true class, columns the predicted class; the diagonal holds the correct predictions. Shown normalised per row, so each row reads as *"where did the cars of this class end up?"*.

In [ ]:
def classification_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


fitted, test_rows = {}, []
for name, model in models.items():
    pipe = build_pipeline(model)
    if name == "Random Forest":
        pipe.set_params(model__n_jobs=-1)   # one model at a time here, so let it use all cores
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    test_rows.append({"Model": name, **classification_metrics(y_test, pipe.predict(X_test))})

test_results = pd.DataFrame(test_rows).set_index("Model").round(3)
test_results

In [ ]:
for name in ["Logistic Regression", "Random Forest"]:
    print(f"===== {name} =====")
    print(classification_report(y_test, fitted[name].predict(X_test), labels=CLASS_ORDER, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name in zip(axes, ["Logistic Regression", "Random Forest"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, fitted[name].predict(X_test), labels=CLASS_ORDER,
        normalize="true", values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{name} — share of each true class")
    ax.grid(False)
plt.tight_layout(); plt.show()

**Interpretation.**

- **Dummy (most frequent)** always answers *Luxury* (the largest class) and is right 26.4% of the time — the level of guessing. Its macro F1 is only 0.104 because it never predicts the other three classes.
- **Logistic Regression** more than doubles that: **57.7% accuracy, macro F1 0.562**. The per-class report shows a very uneven picture: *Low* is easy (F1 0.785, recall 0.82), *Luxury* reasonable (F1 0.618), but *Medium* (0.488) and especially **_High_ (0.355)** are weak. The confusion matrix shows why: **44% of the true _High_ cars are predicted as _Luxury_** and only 31% are found — the linear model pushes most new, more expensive cars into the top class.
- **Random Forest** is better on every metric: **63.7% accuracy, macro F1 0.633**. The gain comes almost entirely from the middle classes: *High* F1 rises from 0.355 to 0.512 and *Medium* from 0.488 to 0.560, while *Low* (0.798) barely changes.
- Cross-validation agrees with the test set (accuracy 0.580 vs 0.577 for logistic regression, 0.635 vs 0.637 for the forest; F1 standard deviation under 0.01), so the comparison is stable, not a lucky split.
- Precision and recall are close to each other for every class with the forest, so it isn't systematically over- or under-predicting any class; logistic regression over-predicts *Luxury* (recall 0.70, precision 0.55).

## 6. Improve and tune the models

`GridSearchCV` tries every combination of the listed hyperparameters with **5-fold cross-validation on the training set** and keeps the one with the best **macro F1**. The test set is only used afterwards, to compare *before vs after*.

- **Logistic Regression — `C`**: the inverse of the regularisation strength. Small `C` = strong regularisation (simpler model), large `C` = weak regularisation.
- **Random Forest** — `n_estimators` (number of trees), `max_features` (share of features each split may consider; lower = more diverse trees) and `min_samples_leaf` (minimum cars per leaf; higher = smoother, less overfitting).

In [ ]:
logreg_search = GridSearchCV(
    build_pipeline(LogisticRegression(max_iter=2000)),
    {"model__C": [0.01, 0.1, 1, 10]},
    cv=5, scoring="f1_macro", n_jobs=-1)
logreg_search.fit(X_train, y_train)
print("Best Logistic Regression:", logreg_search.best_params_, f"| CV macro F1 {logreg_search.best_score_:.3f}")

In [ ]:
rf_search = GridSearchCV(
    build_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    {
        "model__n_estimators": [100, 300],
        "model__max_features": ["sqrt", 0.3],
        "model__min_samples_leaf": [1, 3],
    },
    cv=5, scoring="f1_macro", n_jobs=-1)
rf_search.fit(X_train, y_train)
print("Best Random Forest:", rf_search.best_params_, f"| CV macro F1 {rf_search.best_score_:.3f}")

(pd.DataFrame(rf_search.cv_results_)[["params", "mean_test_score", "mean_fit_time"]]
   .rename(columns={"mean_test_score": "CV macro F1"})
   .sort_values("CV macro F1", ascending=False).round(3))

In [ ]:
searches = {"Logistic Regression": logreg_search, "Random Forest": rf_search}
rows = []
for name, search in searches.items():
    rows.append({"Model": name, "Version": "before", "CV F1": cv_results.loc[name, "CV F1"],
                 **test_results.loc[name].to_dict()})
    rows.append({"Model": f"{name} (tuned)", "Version": "after", "CV F1": search.best_score_,
                 **classification_metrics(y_test, search.best_estimator_.predict(X_test))})

before_after = pd.DataFrame(rows).set_index("Model").round(3)
before_after

**Discussion.**

- **Logistic Regression:** the grid search chose `C = 10` (the weakest regularisation tried), but the gain is negligible: CV macro F1 0.564 → 0.567, test accuracy 0.577 → 0.576 and macro F1 0.562 → 0.563. Regularisation fights *overfitting*, and this model doesn't overfit (train accuracy 0.590 vs test 0.576, section 7.3). Its limit is its **linear decision boundaries**: it can't express interactions such as *"this brand is luxury only when the car is new and low-mileage"*, and tuning `C` can't add that.
- **Random Forest:** best setting `max_features = 0.3`, `min_samples_leaf = 3`, `n_estimators = 300`. CV macro F1 improved from 0.630 to 0.635, test accuracy from 0.637 to **0.642** and macro F1 from 0.633 to **0.639** — a small but consistent gain on both CV and test. Considering more features per split (`0.3` instead of `sqrt`) helped most; `min_samples_leaf = 3` smooths the trees slightly. With `sqrt`, larger leaves made the forest *worse* (0.616–0.618), so the two settings interact. 300 trees scored almost the same as 100 (0.635 vs 0.634) but took about 3.5 times longer to fit, so 100 trees would be the practical choice.
- **Conclusion:** tuning gave the forest about half a percentage point; the linear model stays where it is. The large difference between the two models comes from the model type, not from the hyperparameters.

## 7. Interpret and compare the results

### 7.1 Which classes are predicted well, and which are hard?

In [ ]:
final_models = {"Logistic Regression (tuned)": logreg_search.best_estimator_,
                "Random Forest (tuned)": rf_search.best_estimator_}

per_class = pd.DataFrame({
    name: f1_score(y_test, model.predict(X_test), labels=CLASS_ORDER, average=None)
    for name, model in final_models.items()
}, index=CLASS_ORDER)
ax = per_class.plot(kind="bar", rot=0, figsize=(8, 4))
ax.set(title="F1-score per class (test set)", ylabel="F1", ylim=(0, 1))
plt.tight_layout(); plt.show()
per_class.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, model) in zip(axes, final_models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(X_test), labels=CLASS_ORDER,
        normalize="true", values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
    ax.grid(False)
plt.tight_layout(); plt.show()

# The classes are ordered (Low < Medium < High < Luxury): how many mistakes are only one step off?
rank = {c: i for i, c in enumerate(CLASS_ORDER)}
for name, model in final_models.items():
    pred = pd.Series(model.predict(X_test), index=y_test.index)
    wrong = pred != y_test
    steps = (pred[wrong].map(rank) - y_test[wrong].map(rank)).abs()
    print(f"{name:28s} errors: {wrong.sum():,} of {len(y_test):,} | "
          f"one class off: {(steps == 1).mean():.0%} | two or more classes off: {(steps >= 2).mean():.0%}")

### 7.2 Most influential features — permutation importance
Shuffle one original column at a time in the test set and measure how much **accuracy drops**. Works for both models and on the original columns, so they can be compared directly.

In [ ]:
importance = {}
for name, model in final_models.items():
    result = permutation_importance(model, X_test, y_test, scoring="accuracy",
                                    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    importance[name] = pd.Series(result.importances_mean, index=X_test.columns)

importance = pd.DataFrame(importance).sort_values("Random Forest (tuned)", ascending=False)
importance.plot(kind="barh", figsize=(9, 4.5))
plt.gca().invert_yaxis()
plt.xlabel("Drop in accuracy when the column is shuffled")
plt.title("Permutation importance on the test set")
plt.tight_layout(); plt.show()
importance.round(3)

### 7.3 Is the Random Forest overfitting?

In [ ]:
for name, model in final_models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:28s} train accuracy {train_acc:.3f} | test accuracy {test_acc:.3f} | gap {train_acc - test_acc:.3f}")

### 7.4 Why `PRICE($)` had to go (leakage demonstration)
Same kind of Random Forest, but with only two features: the car type and the price.

In [ ]:
leaky = load_raw_with_price().sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
X_leak, y_leak = leaky[["CAR TYPE", "PRICE($)"]], leaky[TARGET]
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y_leak, test_size=0.2, stratify=y_leak, random_state=RANDOM_STATE)

leaky_pipe = Pipeline([
    ("prep", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["CAR TYPE"])],
                               remainder="passthrough")),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
leaky_pipe.fit(Xl_train, yl_train)
print(f"Accuracy with only CAR TYPE + PRICE($): {accuracy_score(yl_test, leaky_pipe.predict(Xl_test)):.3f}")

With only **two features — the car type and the price — the forest reaches 88% accuracy**, far above the 64% we get from all nine legitimate features together. A simple rule (rank the price among cars of the same type, cut into quarters) already gets 82% without any machine learning (section 1.3). That is the leakage: `PRICE RANGE` is made from the price, so a model given the price just reverses the labelling. It would be useless in practice — the point of the model is to estimate the price tier *before* the price is known. That's why `PRICE($)` is excluded.

### 7.5 Strengths and weaknesses

| | Logistic Regression | Random Forest |
|---|---|---|
| **Accuracy / macro F1 (tuned, test)** | 0.576 / 0.563 | **0.642 / 0.639** |
| **Hard classes** | *High* almost lost (F1 0.36): 43% of *High* cars predicted as *Luxury* | *High* still hardest but much better (F1 0.52) |
| **Interactions between features** | Can't model them — one weight per feature and class | Captured automatically by the tree splits |
| **Overfitting** | None (train 0.590 vs test 0.576) | Clear gap (train 0.821 vs test 0.642), even with `min_samples_leaf = 3` — but the test score is still the best |
| **Interpretability** | Readable weights; probabilities from the softmax | Needs permutation importance to explain |
| **Speed** | A few seconds to train | ≈ 2 minutes per 300-tree fit on one CPU core during tuning |
| **Preprocessing** | Needs scaling and one-hot encoding | Works without scaling |

**Most influential features** (permutation importance, 7.2): for both models **`MILEAGE`** matters most (accuracy drops by 0.30 for logistic regression and 0.16 for the forest when it is shuffled). The forest then relies on **`CAR TYPE`** (0.13), `AGE OF CAR` (0.10) and `BRAND` (0.08). `CAR TYPE` is useless on its own (each type is about 25% per class, 1.3) but important *in combination*, because the class is defined relative to the type — exactly the kind of interaction a linear model can't use. `RATING` and `REVIEW` hardly matter for either model.

### 7.6 Recommendation

We recommend the **tuned Random Forest**.

- **Better overall:** test accuracy **0.642** and macro F1 **0.639**, against 0.576 and 0.563 for logistic regression — it misclassifies 2,865 instead of 3,396 of the 8,000 test cars.
- **Better on the classes that matter most:** the gain is concentrated on the hard middle classes — *High* F1 0.516 vs 0.361, *Medium* 0.559 vs 0.490 — while both models handle *Low* well (0.805 vs 0.783) and *Luxury* reasonably (0.675 vs 0.616).
- **Its mistakes are "near misses":** 81% of its errors are only one price class off (e.g. *High* predicted as *Luxury*), only 19% are two or more classes off. *Low* (cheap, older, high-mileage used cars) is predicted well; *High* is hardest because *High* and *Luxury* cars are both mostly new with zero mileage, so the remaining features (brand, type, origin) have to separate them.
- **Trade-offs:** it is slower and less transparent than logistic regression, and it overfits somewhat (train 0.82 vs test 0.64).

**Limitations and possible improvements:** we used a 40,000-row sample of 240,927 listings and left out the model name and dealer city (thousands of categories). The label itself is relative to the car type, which features alone describe only partly. Next steps: train on all rows, add the model name with rare models grouped, and try gradient boosting. Because the classes are ordered, an ordinal approach (or reporting "within one class" accuracy) would also reflect that a *High*-vs-*Luxury* mistake is less serious than *Low*-vs-*Luxury*.